# 2.3 Applying an ExtraTrees Model for Reservoir Storage Forecasting

**STARS4Water – Training Video 2.3**  
*UK Centre for Ecology and Hydrology*

This notebook accompanies the training video *Machine Learning model for reservoir storage forecasting*, available at the [STARS4Water Academy](https://stars4water.eu/the-stars4water-academy/)

It demonstrates how to apply a pretrained ExtraTrees machine learning model to produce a **1-month ahead reservoir storage forecast**.

**What this notebook covers:**
1. Required packages
2. Load and explore the data
3. Load the pretrained ExtraTrees model
4. Apply the model in simulation mode and inspect the results
5. Load and inspect the ensemble rainfall forecast
6. Apply the model in forecast mode and view results
7. Notes on multi-month recursive forecasting

---
### Running this notebook

**Option 1 – Binder (no installation required)**  
Click the Binder badge in the repository to launch this notebook directly in your browser. All dependencies are installed automatically.

**Option 2 – Local installation**  
Create a virtual environment and install dependencies:

```bash
python3 -m venv venv
source venv/bin/activate          # Windows: venv\Scripts\activate
pip install --upgrade pip
pip install -r requirements.txt
```

Register the kernel and launch Jupyter:

```bash
python -m ipykernel install --user --name=venv --display-name "Python (venv)"
jupyter notebook
```

> **Python version note:** `scikit-learn ≥ 1.7` requires **Python 3.10 or later**.

## 1. Import packages

In [ ]:
import os
import pathlib
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import r2_score, mean_absolute_error

# When launched via Binder/JupyterLab the kernel starts at the repo root.
# Change to the notebook directory so relative paths (data.sav etc.) resolve correctly.
_nb_dir = pathlib.Path('WP3/Task3.4/Reservoir Storage Forecasting')
if _nb_dir.exists():
    os.chdir(_nb_dir)

print("Packages loaded successfully.")

## 2. Load and explore the data

`data.sav` is a pickled pandas DataFrame containing monthly records for **35 reservoirs** across the UK, Spain and France. All storage values are expressed as a **percentage of capacity**.

| Feature group | Features | Description |
|---|---|---|
| **Target** | `storage` | Reservoir storage at time *T* (%) |
| **Lagged storage** | `storage1`, `storage2` | Storage 1 and 2 months ago (%) |
| **Rainfall** | `rainfall`, `rainfall1` | Monthly rainfall (mm): current, 1 month ago |
| **Cumulative rainfall** | `rfquart`, `rfhalf` | Cumulative rainfall over last 3 and 6 months (mm) |
| **Seasonality** | `datesin`, `datecos` | Cyclical month encoding: sin and cos of (2π × month / 12) |
| **Climate** | `swc_pc_s`, `high_prec_freq`, `low_prec_dur` | Soil water content, high precipitation fequency, low precipitation duration |
| **Catchment** | `run_mm_syr`,  `ele_mt_smx` | Long-term runoff, elevation |
| **Reservoir** | `Inception_use_WR/P/Irr/FC`, `Catchment_area`, `Capacity` | Purpose flags, catchment area, capacity |

In [ ]:
DATA_PATH  = 'data.sav'
with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

In [ ]:
# plot monthly storage for example reservoirs
USE_LABELS = {
    'Inception_use_WR':  'Water Resources',
    'Inception_use_P':   'Potable',
    'Inception_use_Irr': 'Irrigation',
    'Inception_use_FC':  'Flood Control',
}

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for ax, res_id in zip(axes, [101, 117, 134]):
    res    = data[data['RESERVOIR_ID'] == res_id].sort_values('date')
    static = res.iloc[0]

    # Primary axis: storage
    ax.fill_between(res['date'], res['storage'], alpha=0.25, color='steelblue')
    ax.plot(res['date'], res['storage'], color='steelblue', linewidth=1.2, label='Storage (%)')
    ax.set_ylabel('Storage (%)', fontsize=10, color='steelblue')
    ax.set_ylim(0, 105)
    ax.tick_params(axis='y', labelcolor='steelblue')
    ax.axhline(50, color='grey', linestyle=':', linewidth=0.8)

    # Static feature annotation
    uses      = [lbl for col, lbl in USE_LABELS.items() if col in static.index and static[col] == 1]
    use_str   = ', '.join(uses) if uses else 'Unknown'
    catch_str = f"{static['Catchment_area']:.0f} km²"
    cap       = static['Capacity']/1000
    cap_str   = f"{cap:.0f} MCM"
    info      = f"Usage: {use_str}   |   Catchment area: {catch_str}   |   Reservoir capacity.: {cap_str}"
    ax.set_title(f"Reservoir {res_id}\n{info}", fontsize=10)

    # Combined legend
    h1, l1 = ax.get_legend_handles_labels()
    ax.legend(h1, l1, fontsize=8, loc='upper left')

axes[-1].set_xlabel('Date', fontsize=10)
plt.suptitle('Monthly Reservoir Storage – Three Example Reservoirs',
             fontsize=12)
plt.tight_layout()
plt.show()

## 3. Load the pretrained ExtraTrees model

`model.sav` is a pickled `scikit-learn` `ExtraTreesRegressor` trained on the multi-reservoir dataset. These are the chosen hyperparameters (see the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesRegressor.html) for more information on hyperparameters).

| Hyperparameter | Value |
|---|---|
| Trees (`n_estimators`) | 200 |
| Max depth | 15 |
| Max features per split | 20 |
| Min samples per leaf | 6 |
| Bootstrap | True |
| Random state | 345 |

In [ ]:
MODEL_PATH = 'model.sav'
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

print(f"Model type      : {type(model).__name__}")
print(f"Trees           : {model.n_estimators}")
print(f"Max depth       : {model.max_depth}")
print(f"Max features    : {model.max_features}")
print(f"Input features  : {model.n_features_in_}")

In [ ]:
feature_names = list(model.feature_names_in_)
importances   = model.feature_importances_
order         = np.argsort(importances)[::-1]
names_sorted  = np.array(feature_names)[order]

def feature_colour(name):
    if 'storage' in name:
        return '#2196F3'
    if 'rain' in name.lower() or name.startswith('rf'):
        return '#FF9800'
    if name in ('datesin', 'datecos'):
        return '#9C27B0'
    return '#4CAF50'

colours = [feature_colour(n) for n in names_sorted]

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(range(len(importances)), importances[order], color=colours, edgecolor='white')
ax.set_xticks(range(len(importances)))
ax.set_xticklabels(names_sorted, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Importance', fontsize=11)
ax.set_title('Feature Importances – ExtraTrees Model', fontsize=12)

legend = [
    mpatches.Patch(color='#2196F3', label='Lagged storage'),
    mpatches.Patch(color='#FF9800', label='Rainfall'),
    mpatches.Patch(color='#9C27B0', label='Seasonality'),
    mpatches.Patch(color='#4CAF50', label='Catchment / climate attributes'),
]
ax.legend(handles=legend, fontsize=9)
plt.tight_layout()
plt.show()

## 4. Apply the model – simulation mode

In **simulation mode** we apply the model to the *historical* record using the fully observed feature set (including observed lagged-storage and rainfall values). This shows how well the model tracks observations over time.

Here we have example data for one reservoir from 1987 to 2022.

In [ ]:
#read the example data
res_data = pd.read_csv('Example_res_prediction.csv')
res_data['date'] = pd.to_datetime(res_data['date'],dayfirst=True)

#split into target and explanatory variables and predict
X_hist = res_data[feature_names]
y_obs  = res_data['storage'].values
y_sim  = model.predict(X_hist)

#calculate metrics
r2  = r2_score(y_obs, y_sim)
mae = mean_absolute_error(y_obs, y_sim)

#print out
print(f"Reservoir: 134")
print(f"Records  : {len(res_data)} monthly observations")
print(f"Period   : {res_data['date'].iloc[0].strftime('%b %Y')} – {res_data['date'].iloc[-1].strftime('%b %Y')}")
print(f"R²       : {r2:.3f}")
print(f"MAE      : {mae:.1f} percentage points")

In [ ]:
#plot observed against predicted
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

# Full record
axes[0].plot(res_data['date'], y_obs, label='Observed',  color='steelblue',  linewidth=1.5)
axes[0].plot(res_data['date'], y_sim, label='Simulated', color='darkorange', linewidth=1.2, linestyle='--')
axes[0].set_ylabel('Storage (%)', fontsize=10)
axes[0].set_title(f'Reservoir 134 – Full record', fontsize=11)
axes[0].set_ylim(0, 105)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.25)

# Zoomed: final 10 years
zoom = res_data[res_data['date'] >= '2013-01-01']
axes[1].plot(zoom['date'], y_obs[-len(zoom):], label='Observed',  color='steelblue',  linewidth=1.8)
axes[1].plot(zoom['date'], y_sim[-len(zoom):], label='Simulated', color='darkorange', linewidth=1.5, linestyle='--')
axes[1].set_ylabel('Storage (%)', fontsize=10)
axes[1].set_title(f'Reservoir 134 – 2013–2022 (zoomed)', fontsize=11)
axes[1].set_ylim(0, 105)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.25)
axes[1].set_xlabel('Date', fontsize=10)

plt.suptitle(f'Simulation vs Observed Storage  |  R² = {r2:.3f},  MAE = {mae:.1f} pp', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Meteorological forecast ensemble

In an operational setting, the rainfall input for the next month must be forecast - here we use an ensemble of rainfall forecasts with 140 members.

In [ ]:
#read and inspect the forecast rainfall for July 2012
rf_forecast = pd.read_csv('Example_July_ensemble.csv')
ens_rf = rf_forecast['rainfall']

forecast_date = pd.Timestamp('2012-07-01')
last_date     = forecast_date - pd.DateOffset(months=1)
fc_month      = forecast_date.month

last_row = res_data[res_data['date'].dt.to_period('M') == last_date.to_period('M')].iloc[0]

print(f"Last observation  : {last_date.strftime('%B %Y')}")
print(f"Forecast target   : {forecast_date.strftime('%B %Y')}")

# Climatological rainfall distribution for the forecast calendar month
hist_rf       = res_data[res_data['date'].dt.month == fc_month]['rainfall']
rf_mean       = hist_rf.mean()
rf_std        = hist_rf.std()

ens_rf_median = ens_rf.median()

# Actual rainfall for the forecast month (if available in res_data)
actual_rf_rows  = res_data[res_data['date'].dt.to_period('M') == forecast_date.to_period('M')]
actual_rf_known = len(actual_rf_rows) > 0
if actual_rf_known:
    actual_rf = actual_rf_rows.iloc[0]['rainfall']

print(f"\nClimatological rainfall for {forecast_date.strftime('%B')}")
print(f"  Historical mean ± std : {rf_mean:.1f} ± {rf_std:.1f} mm")
print(f"  Ensemble median       : {ens_rf_median:.1f} mm")
print(f"  Ensemble range        : {ens_rf.min():.1f} – {ens_rf.max():.1f} mm")
if actual_rf_known:
    print(f"  Actual observed       : {actual_rf:.1f} mm")

# Visualise the ensemble
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(ens_rf, bins=10, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(rf_mean, color='darkorange', linewidth=2, linestyle='--',
           label=f'Climatological mean ({rf_mean:.0f} mm)')
ax.axvline(ens_rf_median, color='purple', linewidth=2, linestyle='--',
           label=f'Ensemble median ({ens_rf_median:.0f} mm)')
ax.axvline(last_row['rainfall'], color='firebrick', linewidth=2, linestyle=':',
           label=f'Last observed ({last_row["rainfall"]:.0f} mm)')
if actual_rf_known:
    ax.axvline(actual_rf, color='#4CAF50', linewidth=2, linestyle='-',
               label=f'Actual ({actual_rf:.0f} mm)')
ax.set_xlabel(f'Forecast Rainfall – {forecast_date.strftime("%B %Y")} (mm)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Rainfall Forecast Ensemble  (n = 140)', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. Apply the model in forecast mode

We have a feature matrix with **one row per ensemble member**. Catchment attributes and most lagged features are held at their last observed values. The features updated for the forecast are:

| Feature | Updated value for forecast at T+1 |
|---|---|
| `datesin`, `datecos` | Cyclical month encoding for forecast month |
| `storage1` | Last observed storage → lag-1 for T+1 |
| `storage2` | `storage1` from current row → lag-2 for T+1 |
| `rainfall` | **Varied across ensemble members** (forecast uncertainty) |
| `rainfall1` | Last observed rainfall → lag-1 for T+1 |

The model is applied independently to all 140 feature rows.

In [ ]:
# Predict
fc_predictions = model.predict(rf_forecast[feature_names])

p10  = np.percentile(fc_predictions, 10)
p50  = np.percentile(fc_predictions, 50)
p90  = np.percentile(fc_predictions, 90)

# Look up actual storage for the forecast month from the full dataset
res_134      = data[data['RESERVOIR_ID'] == 134]
actual_mask  = res_134['date'].dt.to_period('M') == forecast_date.to_period('M')
actual_vals  = res_134.loc[actual_mask, 'storage'].values
actual_known = len(actual_vals) > 0
if actual_known:
    actual_storage = actual_vals[0]
#current storage
fc_storage1 = rf_forecast.storage1[0]

print(f"Forecast: {forecast_date.strftime('%B %Y')}")
print(f"  Current storage          : {fc_storage1:.1f}%")
print(f"  Predicted median         : {p50:.1f}%")
print(f"  10th – 90th percentile   : {p10:.1f}% – {p90:.1f}%")
print(f"  Actual storage           : {actual_storage:.1f}%")

In [ ]:
hist_storage_mean = res_data[res_data['date'].dt.month == fc_month]['storage'].mean()

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(fc_predictions, bins=10, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(hist_storage_mean, color='darkorange', linewidth=2, linestyle='--',
           label=f'Historical mean for {forecast_date.strftime("%B")} ({hist_storage_mean:.1f}%)')
ax.axvline(p50, color='purple', linewidth=2, linestyle='--',
           label=f'Ensemble median ({p50:.1f}%)')
ax.axvline(fc_storage1, color='firebrick', linewidth=2, linestyle=':',
           label=f'Current storage ({fc_storage1:.1f}%)')
if actual_known:
    ax.axvline(actual_storage, color='#4CAF50', linewidth=2, linestyle='-',
               label=f'Actual ({actual_storage:.1f}%)')
ax.set_xlabel(f'Forecast Storage – {forecast_date.strftime("%B %Y")} (%)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title(f'Reservoir 134 – 1-Month Ahead Storage Forecast  (n = 140)', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 7. Multi-month recursive forecasting

For forecast horizons beyond one month, predictions must be made **recursively**:

| Step | Forecast for | `storage1` | `storage2` | `rainfall` |
|---|---|---|---|---|
| 1 | T+1 | Observed at T | Observed at T−1 | 1-month ahead forecast |
| 2 | T+2 | **Predicted** T+1 | Observed at T | 2-month ahead forecast |
| 3 | T+3 | **Predicted** T+2 | **Predicted** T+1 | 3-month ahead forecast |

Each step uses the **previous step's prediction** as the lagged-storage input. Because lagged storage is the dominant predictor (see feature importances above), forecast uncertainty accumulates as the horizon increases. Applying the model to a rainfall ensemble at each step propagates this uncertainty, yielding wider forecast distributions at longer lead times.



> **Tip:** For best performance in your region, consider retraining the model on > local reservoir data — this is demonstrated in the next video in this series.  
> For full technical details, see the **STARS4Water Deliverable D3.4 report** > available on the STARS4Water project website.